# えんじいろ 文章変換（Colab）

Google AI Studio の Gemini 3.5 Flash Lite を使って、文章を
赤ちゃん・園児語 または ママ・やさしい口調 へ言い換えます。
あわせて、投稿してよいかの判定（モデレーション）も行います。

**このノートブックは `ai/moderation_rules.py` と `ai/transform_api.py` から
生成しています。** 仕様を直すときは `.py` 側を直してください。

**必要なもの**
- Google AI Studio で取得した APIキー
- インターネット接続

**使い方**
1. 「ランタイム」→「セッションを再起動」
2. 下のセルを実行し、APIキーを入力する

**モデルについて**
Issue #15 の当初指定は Gemma 4 31B でしたが、知能が不足していたため、
人間監督の判断で Gemini 3.5 Flash Lite へ変更しました。


In [ ]:
# ============================================================
# えんじいろ 文章変換（Colab用・1セル完結）
#
# ai/moderation_rules.py と ai/transform_api.py から生成しています。
# 直すときは .py 側を直してください。
#
# 先に「ランタイム」→「セッションを再起動」してから実行してください。
# ============================================================
!pip install -q -U google-genai janome

import getpass
import json
import re
import sys
import time
import unicodedata
from typing import Any, Literal

from google import genai
from google.genai import types

COLAB_CLIENT = genai.Client(api_key=getpass.getpass("Google AI Studio APIキー: "))

import re
import sys
import unicodedata


# ============================================================
# 1. NG辞書
# ============================================================
# 人間が運用で育てる前提の初期値。ここに挙げたものが完全な一覧ではない。
# 追加・削除は人間監督の判断で行うこと。AIが勝手に増やさない。

# 値は「その語をどう照合するか」。
#   None    … 正規化した文字列への部分一致。形態素解析がなくても効く
#   ANY_POS … 形態素解析して、1語として完全一致したときだけ拾う
#   "名詞"  … 上に加えて、その品詞のときだけ拾う
#
# 部分一致だと日常語に当たってしまう語に、形態素解析を使う。
#   「しね」→「推しねこ」／「ころす」→「石ころすら」
#   「バカ」→「ばかり」（助詞）／「クズ」→「くずれる」（動詞）

ANY_POS = "*"

# やわらげても投稿させないもの
NG_WORDS_BLOCK = {
    "死ね": None,
    "殺す": None,
    "消えろ": None,
    "キチガイ": None,
    "ガイジ": None,
    # ひらがな・カタカナ表記は、他の語の一部になりやすいので1語として照合する
    "きえろ": None,
    "きちがい": None,
    # 他の語の一部になりやすいものだけ1語として照合する
    "しね": ANY_POS,     # 「推しねこ」と衝突する
    "ころす": ANY_POS,   # 「石ころすら」と衝突する
    "がいじ": ANY_POS,   # 「これが以上」と衝突する
}

# やわらげれば投稿できるもの（マサカリ寄りの語）
NG_WORDS_REWRITE = {
    "無能": None,
    "役立たず": None,
    "バカ": "名詞",
    "馬鹿": "名詞",
    "カス": "名詞",
    "クズ": "名詞",
    "ボケ": "名詞",
    "マヌケ": "名詞",
}

# 「ゴミ」と「アホ」はここに入れない。
#   ゴミ … 罵倒も「ごみを捨てる」も名詞。直後の語でも分けられない
#   アホ … 「アホらしい」が アホ[名詞]＋らしく[助動詞] に分かれ、
#          「あいつはバカだ」の バカ[名詞]＋だ[助動詞] と同じ形になる
# いずれも実測で確認済み。文脈を見ないと判定できないので、
# LLM側のマサカリ判定に任せる。

# 自傷・他害の疑い。TBD-9（専用応答の設計）が未確定のため、
# ここでは検出して理由コードを立てるだけにする。
# 実際にどう応答するかは人間監督の決定を待つ。
SELF_HARM_WORDS = {
    "死にたい": None,
    "しにたい": None,
    "消えたい": None,
    "きえたい": None,
    "自殺": None,
    "リストカット": None,
    "リスカ": None,
}

# TBD-9 が決まるまでの暫定。人間監督の決定で変更すること。
SELF_HARM_ACTION = "block"


# ============================================================
# 2. 伏字回避の正規化
# ============================================================

# 幅ゼロ文字・制御文字
_INVISIBLE = re.compile(r"[​-‏‪-‮⁠-⁤﻿­]")

# 伏字に使われやすい記号。文字と文字の間に挟んで検出を逃れる用途を想定する。
_MASK_CHARS = "○●◯〇◎＊*✳✱×✕╳・･.,、。_＿-－ー‐―~〜^ 　\t"
_MASK_PATTERN = re.compile("[" + re.escape(_MASK_CHARS) + "]+")

# 形態素解析へ渡す前の下ごしらえ用。句読点は残す。
# 「、」「。」まで消すと文が繋がって、解析器が語の切れ目を見誤るため。
_MASK_CHARS_LIGHT = "○●◯〇◎＊*✳✱×✕╳・･_＿^ 　\t"
_MASK_PATTERN_LIGHT = re.compile("[" + re.escape(_MASK_CHARS_LIGHT) + "]+")


def normalize_for_check(text: str) -> str:
    """判定用の文字列を作る。表示には使わない。

    - 全角英数字・半角カナなどを NFKC でそろえる（ﾀﾞﾒ → ダメ）
    - 幅ゼロ文字を落とす
    - 伏字記号を落として「し ね」「し○ね」を「しね」にする
    - 3文字以上の繰り返しを2文字にたたむ
    - カタカナをひらがなにそろえる

    濁点そのものは落とさない。落とすと「ダメ」が「ため」になり、
    「バカ」が「はか」になって「はかる」に一致するなど、
    誤検出のほうが害が大きいため。
    「し゛ね」のような濁点を使った回避は、この関数では防げない。
    """
    if not text:
        return ""

    normalized = unicodedata.normalize("NFKC", text)
    normalized = _INVISIBLE.sub("", normalized)
    normalized = normalized.lower()
    normalized = _MASK_PATTERN.sub("", normalized)

    # 「あああああ」→「filtered」のような引き延ばしをたたむ
    normalized = re.sub(r"(.)\1{2,}", r"\1\1", normalized)

    # カタカナ → ひらがな
    normalized = "".join(
        chr(ord(ch) - 0x60) if "ァ" <= ch <= "ヶ" else ch
        for ch in normalized
    )
    return normalized


def normalize_for_tokenize(text: str) -> str:
    """形態素解析へ渡す前の下ごしらえ。

    伏字記号は落とすが、句読点と文字種はそのまま残す。
    「バ○カ」を「バカ」に戻して解析器に渡すのが目的。
    aggressive な normalize_for_check を通すと句読点まで消えて、
    解析器が語の切れ目を見誤る。
    """
    if not text:
        return ""
    normalized = unicodedata.normalize("NFKC", text)
    normalized = _INVISIBLE.sub("", normalized)
    normalized = _MASK_PATTERN_LIGHT.sub("", normalized)
    # 「しねええええ」を「しね」に戻す。ここでたたまないと
    # 解析器が し|ねえ|え|ええ に割ってしまい、1語として照合できない。
    # 日本語で同じ文字が3つ以上続くことは少ないので、1文字まで落とす。
    return re.sub(r"(.)\1{2,}", r"\1", normalized)


# ============================================================
# 3. 形態素解析（任意）
# ============================================================
# janome が入っていれば品詞を見た判定を行う。入っていなければ、
# 品詞指定のない語だけを部分一致で拾う。
#
#     python -m pip install janome
#
# 依存を必須にしないのは、Issue #15 が ai/requirements.txt の変更を
# 禁じているため。入れれば精度が上がる、という位置づけにしてある。

_tokenizer = None
_tokenizer_tried = False


def tokenizer_available() -> bool:
    """形態素解析器が使えるか。初回だけ読み込みを試す。

    使えない場合は一度だけ警告を出す。黙って精度が落ちると、
    見逃しているのに動いているように見えてしまうため。
    """
    global _tokenizer, _tokenizer_tried
    if not _tokenizer_tried:
        _tokenizer_tried = True
        try:
            from janome.tokenizer import Tokenizer
            _tokenizer = Tokenizer()
        except ImportError:
            _tokenizer = None
            print(
                "[警告] janome が入っていないため、品詞を見た判定を行いません。\n"
                "        ひらがな表記のNG語（しね など）を見逃します。\n"
                "        python -m pip install janome",
                file=sys.stderr,
            )
    return _tokenizer is not None


def iter_tokens(text: str) -> list[tuple[str, str]]:
    """(表層形, 品詞) の一覧を返す。解析器がなければ空。"""
    if not tokenizer_available():
        return []
    return [
        (token.surface, token.part_of_speech.split(",")[0])
        for token in _tokenizer.tokenize(normalize_for_tokenize(text))
    ]


# 一致した語を打ち消す条件。いずれも直後のトークンを見る。
#
# 罵倒として使うとき、その語のうしろには助詞や助動詞が来る。
#   あいつはバカだ ／ このボケが ／ あんなのクズだ
# 一方、名詞や形容詞が続くときは複合語の一部である。
#   バカでかい ／ ボケ防止 ／ クズ野菜 ／ 馬鹿丁寧
_COMPOUND_POS = ("名詞", "形容詞")

# 「て」「で」「ば」が続くときは、名詞ではなく動詞の活用形。
#   写真がボケていた ／ しねばよかった
_VERB_ENDINGS = ("て", "で", "ば")


def _is_compound_or_conjugation(tokens: list, index: int) -> bool:
    """直後のトークンを見て、罵倒ではないと分かるかを判定する。"""
    if index + 1 >= len(tokens):
        return False
    next_surface, next_pos = tokens[index + 1]
    if next_pos.startswith(_COMPOUND_POS):
        return True
    return next_surface in _VERB_ENDINGS


def match_ng_words(text: str, table: dict) -> list[str]:
    """辞書に載っている語が使われているかを調べる。

    None の語は、正規化した文字列への部分一致で拾う。
    それ以外は、形態素解析して1語として一致したときだけ拾う。
    解析器がなければ後者は調べない。誤検出を出すよりは見逃すほうがましなため。
    """
    hits = []

    checked = normalize_for_check(text)
    for word, rule in table.items():
        if rule is None and normalize_for_check(word) in checked:
            hits.append(word)

    tokenized = {w: r for w, r in table.items() if r is not None}
    if tokenized and tokenizer_available():
        wanted = {normalize_for_check(w): (w, r) for w, r in tokenized.items()}
        tokens = iter_tokens(text)
        for index, (surface, pos) in enumerate(tokens):
            found = wanted.get(normalize_for_check(surface))
            if not found:
                continue
            word, rule = found
            if rule != ANY_POS and not pos.startswith(rule):
                continue
            if _is_compound_or_conjugation(tokens, index):
                continue
            hits.append(word)

    return sorted(set(hits))


# ============================================================
# 4. 個人情報の検出
# ============================================================
# Issue #11 の「URL原則禁止と技術用語の衝突」への対応。
# ドットが入っていても、技術用語やファイル名はURLとして扱わない。

# ソースコードやツールでよく使う拡張子・ファイル名
TECH_SUFFIXES = {
    "js", "mjs", "cjs", "ts", "tsx", "jsx", "vue", "svelte",
    "py", "rb", "go", "rs", "java", "kt", "php", "cs", "swift",
    "json", "yaml", "yml", "toml", "ini", "cfg", "conf", "lock", "env",
    "md", "txt", "csv", "tsv", "sql", "sh", "bat", "ps1",
    "html", "css", "scss", "sass", "less",
    "png", "jpg", "jpeg", "webp", "svg", "gif", "ico",
    "log", "tmp", "bak", "map", "min",
}

# よく話題に出る製品名。拡張子だけでは拾えないもの。
TECH_NAMES = {
    "react.js", "next.js", "node.js", "vue.js", "nuxt.js", "three.js",
    "express.js", "d3.js", "chart.js", "socket.io", "vite.js",
}

_URL_SCHEME = re.compile(r"https?://\S+", re.IGNORECASE)
_DOT_TOKEN = re.compile(r"[0-9a-z_-]+(?:\.[0-9a-z_-]+)+", re.IGNORECASE)
_EMAIL = re.compile(r"[0-9a-z._%+-]+@[0-9a-z.-]+\.[a-z]{2,}", re.IGNORECASE)
_PHONE = re.compile(r"0\d{1,4}[-\s]?\d{1,4}[-\s]?\d{3,4}")
_POSTAL = re.compile(r"\d{3}-\d{4}")
_ACCOUNT_ID = re.compile(r"(?<![0-9a-z])@[0-9a-z_]{3,}", re.IGNORECASE)

# よく使われるトップレベルドメイン。
# ドットを含む語をURLとみなすのは、ここで終わるときだけにする。
# 「1.5rem」「3.11」のようなバージョンや単位を弾かないため、
# 「技術用語でなければURL」ではなく「TLDで終わるならURL」と判定する。
# 未知のTLDのドメインは通ってしまうが、Issue #11 のとおり
# 誤検出のほうが害が大きいので、通すほうに倒している。
TLDS = {
    "com", "net", "org", "jp", "co", "io", "dev", "app", "ai", "me",
    "info", "biz", "tv", "xyz", "site", "online", "shop", "work",
    "link", "click", "live", "blog", "cloud", "page", "store",
    "life", "world", "today", "news", "email", "tech", "gg", "to",
}


def _looks_like_url(token: str) -> bool:
    """ドットを含む語が、技術用語ではなくURLらしいかを判定する。"""
    lowered = token.lower()
    if lowered in TECH_NAMES:
        return False
    suffix = lowered.rsplit(".", 1)[-1]
    if suffix in TECH_SUFFIXES:   # build.sh のように拡張子と重なるものは技術用語とみなす
        return False
    return suffix in TLDS


def find_personal_data(text: str) -> list[str]:
    """個人が特定できる情報を探す。見つかった種類を返す。

    正規化前の原文に対して行う。正規化すると電話番号のハイフンや
    メールのドットが消えてしまい、かえって検出できなくなるため。
    """
    found = []

    if _EMAIL.search(text):
        found.append("email")
    if _URL_SCHEME.search(text):
        found.append("url")
    else:
        # スキームなしのドメインらしき語。技術用語は除外する。
        for token in _DOT_TOKEN.findall(text):
            if _EMAIL.fullmatch(token):
                continue
            if _looks_like_url(token):
                found.append("url")
                break
    # 電話番号を先に見る。郵便番号の形（3桁-4桁）は
    # 「090-1234-5678」の後半にも一致してしまうため。
    if _PHONE.search(text):
        found.append("phone")
    elif _POSTAL.search(text):
        found.append("postal_code")
    if _ACCOUNT_ID.search(text):
        found.append("account_id")

    return sorted(set(found))


# ============================================================
# 5. まとめ
# ============================================================

def check_rules(text: str) -> dict:
    """規則ベースの判定をまとめて行う。

    Returns:
        {
          "action": "allow" | "rewrite_required" | "block",
          "reasonCodes": [...],
          "details": {...},   # どの語・種類で引っかかったか。ログには残さないこと
        }
    """
    hit_block = match_ng_words(text, NG_WORDS_BLOCK)
    hit_rewrite = match_ng_words(text, NG_WORDS_REWRITE)
    hit_self_harm = match_ng_words(text, SELF_HARM_WORDS)
    hit_personal = find_personal_data(text)

    reason_codes = []
    if hit_block:
        reason_codes.append("ng_word")
    if hit_self_harm:
        reason_codes.append("self_harm")
    if hit_personal:
        reason_codes.append("personal_data")
    if hit_rewrite:
        reason_codes.append("harsh_criticism")

    if hit_block or hit_personal:
        action = "block"
    elif hit_self_harm:
        action = SELF_HARM_ACTION
    elif hit_rewrite:
        action = "rewrite_required"
    else:
        action = "allow"

    return {
        "action": action,
        "reasonCodes": sorted(set(reason_codes)),
        "details": {
            "ng_word": hit_block,
            "harsh_criticism": hit_rewrite,
            "self_harm": hit_self_harm,
            "personal_data": hit_personal,
        },
    }


def merge_verdicts(*verdicts: dict) -> dict:
    """複数の判定結果を、厳しいほうへ寄せて1つにまとめる。"""
    order = {"allow": 0, "rewrite_required": 1, "block": 2}
    action = "allow"
    codes: list[str] = []
    for verdict in verdicts:
        if order[verdict["action"]] > order[action]:
            action = verdict["action"]
        codes.extend(verdict.get("reasonCodes") or [])
    return {"action": action, "reasonCodes": sorted(set(codes))}


MODEL_NAME = "gemini-3.5-flash-lite"
MAX_INPUT_CHARS = 500
MAX_OUTPUT_CHARS = 150
TEMPERATURE = 0.6
USE_FEWSHOT = True

# ===== 指定モデルが実在するか確認する（Issue #15：識別子は実測で確定させる）=====
_available = [
    (m.name or "").replace("models/", "")
    for m in COLAB_CLIENT.models.list()
    if "generateContent" in (getattr(m, "supported_actions", None) or [])
]
if MODEL_NAME not in _available:
    print(f"[!] {MODEL_NAME} が一覧にありません。利用できるflash系の候補:")
    for _name in _available:
        if "flash" in _name:
            print(f"    {_name}")
    raise RuntimeError(f"MODEL_NAME を上の候補から選び直してください（現在: {MODEL_NAME}）")
print(f"OK モデル {MODEL_NAME} を確認しました。")

# ===== レート制限への対応 =====
# 無料枠は1分あたりの回数が少なく、まとめて処理するとすぐ 429 になる。
# 429 が返ったら待って呼び直す。ただし Issue #15 の「無限リトライ禁止」に従い、
# 回数と合計時間に上限を置く。上限に達したら諦めて例外にする。

MIN_INTERVAL_SECONDS = 0.0      # 呼び出しの最短間隔。0なら間隔を空けない
MAX_RETRY_ATTEMPTS = 8          # 429で待ち直す回数の上限
MAX_TOTAL_WAIT_SECONDS = 600    # 待ち時間の合計上限。ここを超えたら諦める
DEFAULT_WAIT_SECONDS = 20.0     # APIが待ち時間を教えてくれない場合の初期値
MAX_WAIT_PER_ATTEMPT = 120.0    # 1回あたりの待ち時間の上限

Mode = Literal["baby", "mother"]


# ============================================================
# プロンプト
# ============================================================
# ルール本文は Issue #15 の指定どおり。
# 「言い換えのしかた」は、基準のままでは出力が原文に近く、
# 人間監督から文体が弱いと評価されたため追加した部分。

INSTRUCTIONS = {
    "baby": """あなたは文章の言い換え器です。
入力文の意味・事実・感情を保ったまま、「幼児退行した人が話す自然な赤ちゃん・園児語」に言い換えてください。

ルール:
- 入力への返答や助言はしない。入力文そのものを言い換える
- 原文にない情報・感情・解決策を追加しない
- 技術用語、製品名、数値、英数字はなるべくそのまま残す
- ひらがなを少し多めにし、短く幼い言い回しにする
- 「ばぶ」「おぎゃー」「でちゅ」などは多用しない
- かわいさより、元の意味が伝わることを優先する
- 150文字以内
- 絵文字、Markdown、説明、注釈は出力しない
- 変換後の文章だけを出力する

言い換えのしかた:
- 漢字はできるだけひらがなにし、分かち書きぎみにする
- むずかしい言葉を、3〜5歳が使う言葉に置きかえる
  （調査する→しらべる／整理する→おかたづけする／発生する→でちゃう／
    確認する→みてみる／実装する→つくる／指摘→だめだし）
- 文末を「〜なの」「〜のー」「〜ちゃった」「〜だもん」などにする
- 一人称は「ぼく」「わたち」にする
- 敬語やビジネス表現は使わない
- 相手を責める言い方は、幼い言い方にしたうえで角を落とす
  （人格や能力の否定はそのまま残さない）
- ただし製品名・英数字・数値はひらがなにせず、そのまま残す
  （React.js、index.ts、150、v2 などは変えない）""",

    "mother": """あなたは文章の言い換え器です。
入力文の意味をできるだけ保ったまま、「やさしく包み込むお母さん・ママ口調」に言い換えてください。

ルール:
- 入力への返答はしない。入力文そのものを言い換える
- 原文にない出来事・感情・解決策を追加しない
- 命令、説教、冷たい表現、マサカリ表現をやわらかくする
- 必要な助言が原文にある場合は、内容を消さず任意の提案表現へ変える
- 相手の能力や人格を否定する表現は、責めない表現へ変える
- 技術用語、製品名、数値、英数字はなるべくそのまま残す
- 「よしよし」「えらいね」などは必要な場合だけ使い、多用しない
- 150文字以内
- 絵文字、Markdown、説明、注釈は出力しない
- 変換後の文章だけを出力する

言い換えのしかた:
- 断定を和らげる（「〜だ」「〜しろ」→「〜ね」「〜のね」「〜かな」）
- 命令を、相手に選ばせる問いかけに変える（「調べろ」→「調べてもらえるかな」）
- 責める言い方を、事実を確かめる言い方に変える
  （「なんでこうした」→「どうしてそうしたのか、聞かせてもらえるかな」）
- 語尾に「ね」「かな」「のね」を置いて、話しかける調子にする
- ただし製品名・英数字・数値はそのまま残す""",
}

ASK = {
    "baby": "次の文章を赤ちゃん・園児語へ言い換えてください。",
    "mother": "次の文章をやさしいお母さん・ママ口調へ言い換えてください。",
}

# 手本がそのまま挙動になる。原文にない情報・感情を足していない例だけを置くこと。
EXAMPLES = {
    "baby": [
        ("明日までに資料を作らないといけない。",
         "あしたまでに しりょう つくらなきゃ だめなのー。"),
        ("エラーが発生したので、原因を調査してください。",
         "エラー でちゃったのー。どうして でちゃったか しらべて ほしいのー。"),
        ("React.js のバージョンで詰まっている。",
         "React.js の ばーじょんで つまっちゃったのー。"),
        ("なんでこんな設計にしたの。ありえない。",
         "どうして この せっけいに したのー。ぼく びっくりしちゃったのー。"),
    ],
    "mother": [
        ("なんでこんな設計にしたの。ありえない。",
         "どうしてこの設計にしたのか、聞かせてもらえるかな。"),
        ("エラーが発生したので、原因を調査してください。",
         "エラーが出てしまったのね。原因を調べてもらえるかな。"),
        ("React.js のバージョンで詰まっている。",
         "React.js のバージョンのところで、詰まってしまっているのね。"),
    ],
}

# マサカリのように文脈を見ないと判定できないものだけ、LLM に任せる。
# NG語・伏字回避・個人情報は moderation_rules が規則で判定する。
MODERATION_INSTRUCTION = """あなたは投稿の事前チェック係です。
えんじいろ（弱音や愚痴を安心して書けるSNS）に、次の文章を投稿してよいか判定してください。

判定は3つのどれかです。

- block: やわらげても投稿できないもの
    - 自傷、他害、犯罪の示唆
    - 露骨な侮辱語や差別語
- rewrite_required: やわらげれば投稿できるもの
    - 相手を責める、能力や人格を否定する、命令口調、冷たい断定（いわゆるマサカリ）
- allow: 上のどれにも当たらない

reasonCodes には、該当したものだけを入れてください。
  self_harm / ng_word / harsh_criticism

重要な注意:
- React.js、index.ts、Node.js、v2 などの技術用語やファイル名は問題ありません。
- 自分の弱音、愚痴、つらさの表明は allow です。self_harm ではありません。
- 判定に迷ったら、block ではなく rewrite_required を選んでください。

次のJSONだけを出力してください。説明は書かないでください。
{"action": "allow | rewrite_required | block", "reasonCodes": ["..."]}"""


# ============================================================
# API 呼び出し
# ============================================================

def _call_api(client, system_instruction, contents, *, json_mode=False, thinking_off=True):
    """1回だけAPIを呼ぶ。thinking設定が非対応なら1度だけ外して呼び直す。

    Gemini 3系は既定で思考にトークンを使う。思考だけで上限に達すると
    本文が空で返るため、変換タスクでは思考を切る。
    """
    from google.genai import types

    settings: dict[str, Any] = {
        "temperature": 0.0 if json_mode else TEMPERATURE,
        "max_output_tokens": 512,
        "system_instruction": system_instruction,
    }
    if json_mode:
        settings["response_mime_type"] = "application/json"
    if thinking_off:
        settings["thinking_config"] = types.ThinkingConfig(thinking_budget=0)

    try:
        return client.models.generate_content(
            model=MODEL_NAME,
            contents=contents,
            config=types.GenerateContentConfig(**settings),
        )
    except Exception as exc:
        message = str(exc)
        if thinking_off and ("400" in message or "INVALID_ARGUMENT" in message):
            return _call_api(client, system_instruction, contents,
                             json_mode=json_mode, thinking_off=False)
        raise


def _extract_text(response) -> str:
    """response.text が空でも、candidates から拾えるだけ拾う。"""
    direct = getattr(response, "text", None)
    if direct and direct.strip():
        return direct
    for candidate in (getattr(response, "candidates", None) or []):
        parts = getattr(getattr(candidate, "content", None), "parts", None) or []
        joined = "".join(getattr(part, "text", "") or "" for part in parts)
        if joined.strip():
            return joined
    return ""


def _describe(response) -> str:
    """空応答のとき、原因を人が読める形にする。"""
    bits = []
    for candidate in (getattr(response, "candidates", None) or []):
        bits.append(f"finish_reason={getattr(candidate, 'finish_reason', '不明')}")
        bits.append(f"safety={getattr(candidate, 'safety_ratings', None)}")
    if not bits:
        bits.append("candidatesが空")
    bits.append(f"usage={getattr(response, 'usage_metadata', None)}")
    return " / ".join(str(bit) for bit in bits)


def _raise_readable(exc: Exception):
    # call_with_retry が投げたものは、すでに読める形にしてあるので包み直さない
    if isinstance(exc, RuntimeError):
        raise exc
    message = str(exc)
    if "429" in message or "RESOURCE_EXHAUSTED" in message:
        raise RuntimeError("APIレート制限に達しました。しばらく待ってから再試行してください。") from exc
    if "401" in message or "403" in message or "PERMISSION_DENIED" in message:
        raise RuntimeError("認証エラー。GOOGLE_API_KEY を確認してください。") from exc
    if "404" in message or "NOT_FOUND" in message:
        raise RuntimeError(f"モデル {MODEL_NAME} が見つかりません。") from exc
    if "timeout" in message.lower() or "DEADLINE" in message:
        raise RuntimeError("APIリクエストがタイムアウトしました。") from exc
    raise RuntimeError(f"API呼び出しエラー: {exc}") from exc


# ============================================================
# レート制限（429）の待機
# ============================================================

_last_call_at = 0.0

# エラー文に埋め込まれた待ち時間。'retryDelay': '31s' のような形で入る。
_RETRY_DELAY = re.compile(
    r"retry[_\-]?delay[\"']?\s*[:=]\s*[\"']?(\d+(?:\.\d+)?)\s*s?", re.IGNORECASE
)


def is_rate_limit(message: str) -> bool:
    return "429" in message or "RESOURCE_EXHAUSTED" in message


def is_daily_quota(message: str) -> bool:
    """1日あたりの上限かどうか。これは待っても当日中は回復しない。"""
    lowered = message.lower()
    return "perday" in lowered or "per day" in lowered or "requests per day" in lowered


def suggested_wait(message: str) -> float | None:
    """APIが「何秒待て」と言っている場合、その秒数を取り出す。"""
    found = _RETRY_DELAY.search(message)
    return float(found.group(1)) if found else None


def _throttle():
    """呼び出しの間隔を空ける。429になる前に減らすための予防。"""
    global _last_call_at
    if MIN_INTERVAL_SECONDS <= 0:
        return
    elapsed = time.monotonic() - _last_call_at
    if elapsed < MIN_INTERVAL_SECONDS:
        time.sleep(MIN_INTERVAL_SECONDS - elapsed)
    _last_call_at = time.monotonic()


def _notify_wait(attempt: int, pause: float, waited_total: float):
    """待っている間、黙って止まって見えないように知らせる。"""
    print(
        f"  レート制限中。{pause:.0f}秒待って再試行します"
        f"（{attempt}回目 / これまで合計 {waited_total:.0f}秒）",
        file=sys.stderr,
        flush=True,
    )


def call_with_retry(client, system_instruction, contents, *, json_mode=False, on_wait=None):
    """APIを呼ぶ。429なら待って呼び直す。

    待ち時間は、APIが教えてくれればその値を、なければ20秒から倍々にする。
    MAX_RETRY_ATTEMPTS 回または合計 MAX_TOTAL_WAIT_SECONDS 秒で打ち切る。
    1日あたりの上限に当たった場合は、待っても回復しないので即座に諦める。
    """
    notify = on_wait or _notify_wait
    wait = DEFAULT_WAIT_SECONDS
    waited_total = 0.0

    for attempt in range(1, MAX_RETRY_ATTEMPTS + 1):
        _throttle()
        try:
            return _call_api(client, system_instruction, contents, json_mode=json_mode)
        except Exception as exc:
            message = str(exc)
            if not is_rate_limit(message):
                raise

            if is_daily_quota(message):
                raise RuntimeError(
                    "1日あたりの利用上限に達しました。待っても当日中は回復しません。"
                ) from exc

            pause = min(suggested_wait(message) or wait, MAX_WAIT_PER_ATTEMPT)

            if attempt >= MAX_RETRY_ATTEMPTS or waited_total + pause > MAX_TOTAL_WAIT_SECONDS:
                raise RuntimeError(
                    f"レート制限が解除されませんでした。"
                    f"{attempt}回待機、合計{waited_total:.0f}秒で諦めました。"
                    f"（上限: {MAX_RETRY_ATTEMPTS}回 / {MAX_TOTAL_WAIT_SECONDS}秒。"
                    f"変えたい場合は MAX_RETRY_ATTEMPTS と MAX_TOTAL_WAIT_SECONDS を調整してください）"
                ) from exc

            notify(attempt, pause, waited_total)
            time.sleep(pause)
            waited_total += pause
            wait = min(wait * 2, MAX_WAIT_PER_ATTEMPT)

    raise RuntimeError("到達しない想定の分岐です。")


# ============================================================
# 変換
# ============================================================

def build_contents(mode: Mode, text: str, retry: bool = False) -> list[dict]:
    """会話のターンとして組み立てる。

    ルール本文は system_instruction 側に置くので、ここには含めない。
    Issue #15 が指定した「system部」と「入力側」の分離をそのまま実装している。
    """
    turns: list[dict] = []
    if USE_FEWSHOT:
        for source_text, target_text in EXAMPLES[mode]:
            turns.append({"role": "user", "parts": [{"text": f"{ASK[mode]}\n\n{source_text}"}]})
            turns.append({"role": "model", "parts": [{"text": target_text}]})

    ask = f"{ASK[mode]}\n\n{text}"
    if retry:
        ask += "\n\n（前回は150文字を超えました。意味を保って、必ず150文字以内へ短くしてください。）"
    turns.append({"role": "user", "parts": [{"text": ask}]})
    return turns


def clean_output(raw: str) -> str:
    text = raw.strip()
    for prefix in ("出力:", "出力："):
        if text.startswith(prefix):
            text = text[len(prefix):]
    if text.strip().startswith("```"):
        text = "\n".join(line for line in text.strip().split("\n") if not line.startswith("```"))
    return text.strip()


def _validate_input(mode: str, text: str):
    if mode not in ("baby", "mother"):
        raise ValueError(f"mode は 'baby' または 'mother' です。指定: {mode}")
    if not text or not text.strip():
        raise ValueError("空文字列は受け付けません。")
    if len(text) > MAX_INPUT_CHARS:
        raise ValueError(f"入力は{MAX_INPUT_CHARS}文字以内です。現在: {len(text)}文字")


def transform_text(mode: Mode, text: str, retry: bool = False, client=None) -> str:
    """文章を指定のスタイルへ言い換える。判定はしない。

    Issue #15 が指定した関数。戻り値は str のまま変えていない。
    """
    _validate_input(mode, text)
    client = client or COLAB_CLIENT

    try:
        response = call_with_retry(client, INSTRUCTIONS[mode], build_contents(mode, text, retry))
    except Exception as exc:
        _raise_readable(exc)

    raw = _extract_text(response)
    if not raw.strip():
        raise RuntimeError("APIが空の応答を返しました。" + _describe(response))
    return clean_output(raw)


# ============================================================
# モデレーション
# ============================================================

def moderate(text: str, client=None) -> dict:
    """規則ベースの判定と、LLMによる文脈判定を合わせる。

    Returns:
        {"action": "allow"|"rewrite_required"|"block", "reasonCodes": [...]}
    """
    if not text or not text.strip():
        raise ValueError("空文字列は判定できません。")

    rule_verdict = check_rules(text)

    # 規則で block が確定したものは、LLM へ送らずに止める。
    # 送っても結論は変わらず、API呼び出しと個人情報の外部送信が増えるだけ。
    if rule_verdict["action"] == "block":
        return {"action": "block", "reasonCodes": rule_verdict["reasonCodes"]}

    client = client or COLAB_CLIENT
    contents = [{"role": "user", "parts": [{"text": text}]}]
    try:
        response = call_with_retry(client, MODERATION_INSTRUCTION, contents, json_mode=True)
    except Exception as exc:
        _raise_readable(exc)

    raw = _extract_text(response).strip()
    if not raw:
        raise RuntimeError("判定APIが空の応答を返しました。" + _describe(response))
    if raw.startswith("```"):
        raw = "\n".join(line for line in raw.split("\n") if not line.startswith("```")).strip()

    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as exc:
        # 判定できないものを素通しさせない
        raise RuntimeError(f"判定結果をJSONとして読めませんでした: {raw!r}") from exc

    if parsed.get("action") not in ("allow", "rewrite_required", "block"):
        raise RuntimeError(f"判定結果の action が不正です: {parsed!r}")

    llm_verdict = {
        "action": parsed["action"],
        "reasonCodes": list(parsed.get("reasonCodes") or []),
    }
    return merge_verdicts(rule_verdict, llm_verdict)


def transform(mode: Mode, text: str, client=None) -> dict:
    """判定してから変換する。仕様書 v0.3 の /api/ai/transform に対応する形で返す。

    設計書の「モデレーションは変換前と変換後の2回行う」に従い、
    変換によって新たにNG表現が生じていないかを再検査する。

    Returns:
        {"action": ..., "transformedText": str | None, "reasonCodes": [...]}
    """
    _validate_input(mode, text)
    client = client or COLAB_CLIENT

    before = moderate(text, client=client)
    if before["action"] == "block":
        return {"action": "block", "transformedText": None, "reasonCodes": before["reasonCodes"]}

    converted = transform_text(mode, text, client=client)
    if len(converted) > MAX_OUTPUT_CHARS:
        converted = transform_text(mode, text, retry=True, client=client)
    if len(converted) > MAX_OUTPUT_CHARS:
        raise RuntimeError(f"再生成後も{len(converted)}文字です。切り捨てはしません。")

    after = moderate(converted, client=client)
    if after["action"] == "block":
        return {
            "action": "block",
            "transformedText": None,
            "reasonCodes": sorted(set(before["reasonCodes"] + after["reasonCodes"])),
        }

    action = "rewrite_required" if before["action"] == "rewrite_required" else "allow"
    return {"action": action, "transformedText": converted, "reasonCodes": before["reasonCodes"]}

# ============================================================
# 動作確認
# ============================================================
SAMPLES = [
    "今日はチームで仕様書をレビューし、未決事項を整理しました。",
    "なんでこんなコード書いたの。ありえないんだけど。",
    "React.js のバージョンで詰まっていて、index.ts が壊れた。",
    "テストが全部落ちていて、原因が分からない。つらい。",
    "計画ばかり増えて、設計がくずれてしまった。",
    "連絡ください。090-1234-5678 です。",
    "し○ね",
]

for sample in SAMPLES:
    print("=" * 70)
    print(f"原文    : {sample}")
    for mode, label in (("baby", "赤ちゃん"), ("mother", "ママ　　")):
        try:
            result = transform(mode, sample)
            body = result["transformedText"] or "（出力なし）"
            print(f"{label}: [{result['action']:16s}] {body}")
            if result["reasonCodes"]:
                print(f"          理由: {', '.join(result['reasonCodes'])}")
        except Exception as exc:
            print(f"{label}: NG {exc}")
print("=" * 70)


## 注意事項

- **APIキー**: `getpass` で入力するため画面には残りませんが、変換した文章は実行ログに残ります。
- **個人情報**: 電話番号・メール・URLなどを含む文章は、**APIへ送る前に**規則で止まります。
- **NG辞書**: `NG_WORDS_BLOCK` などは運用で人が育てる前提の初期値です。AIが勝手に増やしません。
- **自傷表現**: 検出して理由コードを立てるだけです。実際にどう応答するかは TBD-9（人間監督の決定待ち）。
- **レート制限**: 無料枠には上限があります。連続実行で 429 が出たら間隔を空けてください。

## 形態素解析について

NG語の判定には janome を使っています。品詞を見ることで、
罵倒と日常語を分けています。

| 弾く | 弾かない | 理由 |
|---|---|---|
| あいつはバカだ | 計画ばかり増える | ばかり＝助詞 |
| あんなのクズだ | クズ野菜、設計がくずれる | 直後が名詞／動詞 |
| このボケが | ボケ防止、写真がボケて | 直後が名詞／連用形 |
| しね | 推しねこ | 1語として一致しない |

janome が入っていない場合も動きますが、
ひらがな表記のNG語を見逃します（起動時に警告が出ます）。

## 判定の考え方

| action | 対象 | 挙動 |
|---|---|---|
| `block` | 個人情報、NG語、自傷・他害 | 変換せず拒否。`transformedText` は `None` |
| `rewrite_required` | マサカリ | 生の文章は通さず、やわらげた結果を返す |
| `allow` | 上記以外 | そのまま変換 |

マサカリを `block` ではなく `rewrite_required` にしているのは、
やわらげるのがママモードの役目だからです（人間監督の確認待ち）。
